# Lab Assignment 2 - Part B: k-Nearest Neighbor Classification
Please refer to the `README.pdf` for full laboratory instructions.


## Problem Statement
In this part, you will implement the k-Nearest Neighbor (k-NN) classifier and evaluate it on two datasets:
- **Lenses Dataset**: A small dataset for contact lens prescription
- **Credit Approval (CA) Dataset**: Credit card application data with binary labels (+/-)

### Your Tasks
1. **Preprocess the data**: Handle missing values and normalize features
2. **Implement k-NN** with L2 distance
3. **Evaluate** on both datasets for different values of k
4. **Discuss** your results

### Datasets
The data files are located in the `credit 2017/` folder:
- `lenses.training`, `lenses.testing`
- `crx.data.training`, `crx.data.testing`
- `crx.names` (describes the features)


## Setup


In [ ]:
# Library declarations
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter


In [ ]:
# Data paths
DATA_PATH = "credit 2017/"

# Load Lenses data
def load_lenses_data():
    """Load the lenses dataset."""
    train_data = np.loadtxt(DATA_PATH + "lenses.training", delimiter=',')
    test_data  = np.loadtxt(DATA_PATH + "lenses.testing",  delimiter=',')

    # First column is ID, last column is label
    X_train = train_data[:, 1:-1]
    y_train = train_data[:, -1]
    X_test  = test_data[:, 1:-1]
    y_test  = test_data[:, -1]

    return X_train, y_train, X_test, y_test


def load_credit_data(filepath):
    """
    Load raw rows from the Credit Approval dataset.
    Returns a list of lists (all values as strings).
    Missing values appear as '?'.
    """
    rows = []
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(line.split(','))
    return rows


# Test loading lenses data
X_train_lenses, y_train_lenses, X_test_lenses, y_test_lenses = load_lenses_data()
print(f"Lenses - Train: {X_train_lenses.shape}, Test: {X_test_lenses.shape}")


## Task 1: Data Preprocessing
For the Credit Approval dataset, you need to:
1. **Handle missing values** (marked with '?'):
   - Categorical features: replace with mode/median
   - Numerical features: replace with label-conditioned mean
2. **Normalize features** using z-scaling:
   $$z_i^{(m)} = \frac{x_i^{(m)} - \mu_i}{\sigma_i}$$

Document exactly how you handle each feature!


In [ ]:
# Feature column indices based on crx.names (0-indexed, 15 feature columns total)
# Categorical: A1, A4, A5, A6, A7, A9, A10, A12, A13
CAT_COLS = [0, 3, 4, 5, 6, 8, 9, 11, 12]
# Numerical:  A2, A3, A8, A11, A14, A15
NUM_COLS = [1, 2, 7, 10, 13, 14]


def preprocess_credit_data(train_file, test_file):
    """
    Preprocess the Credit Approval dataset.

    Steps:
    1. Load and parse the data (comma-separated, '?' = missing)
    2. Separate features (A1-A15) from labels (A16: + / -)
    3. Impute missing values:
       - Categorical features: replace with mode of that column (training set)
       - Numerical features:   replace with label-conditioned mean (training set)
    4. Encode categorical features to integer codes
    5. Z-normalize numerical features using training statistics

    Returns:
    --------
    X_train, y_train, X_test, y_test : numpy arrays
    """
    # ---- Load raw rows ----
    def read_file(path):
        rows = []
        with open(path, 'r') as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(line.split(','))
        return rows

    train_rows = read_file(train_file)
    test_rows  = read_file(test_file)

    # Separate labels and features; convert '?' to None
    def parse(rows):
        feats  = [[v.strip() if v.strip() != '?' else None for v in r[:-1]] for r in rows]
        labels = np.array([1 if r[-1].strip() == '+' else 0 for r in rows])
        return feats, labels

    train_feat, y_train = parse(train_rows)
    test_feat,  y_test  = parse(test_rows)

    # ---- Build categorical encoders (mode + integer mapping) from training data ----
    cat_encoders = {}
    for col in CAT_COLS:
        vals = [row[col] for row in train_feat if row[col] is not None]
        mode_val = Counter(vals).most_common(1)[0][0]
        encoder  = {v: i for i, v in enumerate(sorted(set(vals)))}
        cat_encoders[col] = {'mode': mode_val, 'encoder': encoder}

    # ---- Build label-conditioned mean imputers for numerical columns ----
    num_imputers = {}
    for col in NUM_COLS:
        pos = [float(r[col]) for r, lbl in zip(train_feat, y_train) if r[col] is not None and lbl == 1]
        neg = [float(r[col]) for r, lbl in zip(train_feat, y_train) if r[col] is not None and lbl == 0]
        all_vals = [float(r[col]) for r in train_feat if r[col] is not None]
        num_imputers[col] = {
            'mean_pos': np.mean(pos)      if pos      else np.mean(all_vals),
            'mean_neg': np.mean(neg)      if neg      else np.mean(all_vals),
            'mean_all': np.mean(all_vals) if all_vals else 0.0,
        }

    # ---- Apply imputation + encoding ----
    def process_rows(arr, labels=None):
        X = np.zeros((len(arr), 15))
        for i, row in enumerate(arr):
            for col in CAT_COLS:
                val = row[col] if row[col] is not None else cat_encoders[col]['mode']
                X[i, col] = cat_encoders[col]['encoder'].get(val, 0)
            for col in NUM_COLS:
                if row[col] is not None:
                    X[i, col] = float(row[col])
                elif labels is not None:
                    X[i, col] = num_imputers[col]['mean_pos' if labels[i] == 1 else 'mean_neg']
                else:
                    X[i, col] = num_imputers[col]['mean_all']
        return X

    X_train_raw = process_rows(train_feat, y_train)
    X_test_raw  = process_rows(test_feat)   # test labels unknown during imputation

    # ---- Z-normalize numerical features using training statistics ----
    X_train = X_train_raw.copy()
    X_test  = X_test_raw.copy()
    norm_stats = {}
    for col in NUM_COLS:
        mu  = np.mean(X_train_raw[:, col])
        sig = np.std(X_train_raw[:, col])
        if sig < 1e-8:
            sig = 1.0
        X_train[:, col] = (X_train_raw[:, col] - mu) / sig
        X_test[:, col]  = (X_test_raw[:, col]  - mu) / sig
        norm_stats[col] = (mu, sig)

    # Summary report
    print("Credit Approval preprocessing complete:")
    print(f"  Train: {X_train.shape}, Test: {X_test.shape}")
    print(f"  Class distribution (train): +={np.sum(y_train==1)}, -={np.sum(y_train==0)}")
    print("\nMissing value imputation:")
    print(f"  Categorical {[f'A{c+1}' for c in CAT_COLS]}: mode imputation")
    for col in CAT_COLS:
        mode = cat_encoders[col]['mode']
        print(f"    A{col+1}: mode = '{mode}'")
    print(f"  Numerical {[f'A{c+1}' for c in NUM_COLS]}: label-conditioned mean")
    for col in NUM_COLS:
        print(f"    A{col+1}: μ(+)={num_imputers[col]['mean_pos']:.3f}, μ(-)={num_imputers[col]['mean_neg']:.3f}")
    print("\nZ-normalization (numerical features, training stats):")
    for col in NUM_COLS:
        mu, sig = norm_stats[col]
        print(f"    A{col+1}: μ={mu:.4f}, σ={sig:.4f}")

    return X_train, y_train, X_test, y_test


def z_normalize(X_train, X_test, feature_indices):
    """
    Apply z-score normalization to specified feature indices.
    Uses training statistics for both train and test.
    """
    X_train_n = X_train.copy().astype(float)
    X_test_n  = X_test.copy().astype(float)
    for i in feature_indices:
        mu  = np.mean(X_train[:, i])
        sig = np.std(X_train[:, i])
        if sig < 1e-8:
            sig = 1.0
        X_train_n[:, i] = (X_train[:, i] - mu) / sig
        X_test_n[:, i]  = (X_test[:, i]  - mu) / sig
    return X_train_n, X_test_n


## Task 2: Implement k-NN Classifier
Implement k-NN with L2 (Euclidean) distance:
$$\mathcal{D}_{L2}(\mathbf{a}, \mathbf{b}) = \sqrt{\sum_i (a_i - b_i)^2}$$

For **categorical attributes**, use:
- Distance = 1 if values are different
- Distance = 0 if values are the same


In [ ]:
def l2_distance(a, b):
    """
    Compute L2 (Euclidean) distance between two vectors.

    Parameters:
    -----------
    a, b : numpy arrays of the same shape

    Returns:
    --------
    distance : float
    """
    return np.sqrt(np.sum((a - b) ** 2))


def mixed_distance(a, b, cat_cols, num_cols):
    """
    Combined distance for mixed-type feature vectors:
      - Numerical  features: squared difference (L2 contribution)
      - Categorical features: 1 if values disagree, 0 if they agree
    Total = sqrt(sum of all contributions)
    """
    dist_sq = 0.0
    for i in num_cols:
        dist_sq += (a[i] - b[i]) ** 2
    for i in cat_cols:
        dist_sq += 0.0 if a[i] == b[i] else 1.0
    return np.sqrt(dist_sq)


def knn_predict(X_train, y_train, X_test, k, cat_cols=None, num_cols=None):
    """
    Predict labels for test data using k-NN.

    Parameters:
    -----------
    X_train  : numpy array (n_train, n_features)
    y_train  : numpy array (n_train,)
    X_test   : numpy array (n_test,  n_features)
    k        : int, number of neighbors
    cat_cols : list of categorical feature indices (None → none)
    num_cols : list of numerical  feature indices  (None → all)

    Returns:
    --------
    predictions : numpy array (n_test,)
    """
    if cat_cols is None:
        cat_cols = []
    if num_cols is None:
        num_cols = list(range(X_train.shape[1]))

    use_mixed = len(cat_cols) > 0
    predictions = []

    for test_pt in X_test:
        if use_mixed:
            dists = np.array([mixed_distance(test_pt, tr_pt, cat_cols, num_cols)
                              for tr_pt in X_train])
        else:
            # Pure vectorized L2
            dists = np.sqrt(np.sum((X_train - test_pt) ** 2, axis=1))

        k_idx   = np.argsort(dists)[:k]
        k_labels = y_train[k_idx]
        vote    = Counter(k_labels).most_common(1)[0][0]
        predictions.append(vote)

    return np.array(predictions)


def compute_accuracy(y_true, y_pred):
    """
    Compute classification accuracy.

    Returns:
    --------
    accuracy : float (0 to 1)
    """
    return float(np.mean(y_true == y_pred))


## Task 3: Evaluate on Lenses Dataset
Test your k-NN implementation on the Lenses dataset for different values of k.


In [ ]:
# All lenses features are categorical (ordinal codes: 1, 2, 3)
LENSES_CAT_COLS = list(range(X_train_lenses.shape[1]))  # e.g. [0, 1, 2]
LENSES_NUM_COLS = []

k_values       = [1, 3, 5, 7]
lenses_results = []

print("k-NN on Lenses Dataset (all features categorical):")
print(f"  {'k':>3}  {'Accuracy':>10}  {'Correct/Total':>15}")
print("  " + "-" * 32)

for k in k_values:
    preds   = knn_predict(X_train_lenses, y_train_lenses, X_test_lenses, k,
                          cat_cols=LENSES_CAT_COLS, num_cols=LENSES_NUM_COLS)
    acc     = compute_accuracy(y_test_lenses, preds)
    correct = int(round(acc * len(y_test_lenses)))
    lenses_results.append((k, acc))
    print(f"  {k:>3}  {acc:>10.4f}  {correct:>7}/{len(y_test_lenses):<7}")


## Task 4: Evaluate on Credit Approval Dataset
First preprocess the data, then evaluate k-NN.


In [ ]:
DATA_PATH = "credit 2017/"
X_train_credit, y_train_credit, X_test_credit, y_test_credit = preprocess_credit_data(
    DATA_PATH + "crx.data.training",
    DATA_PATH + "crx.data.testing"
)


In [ ]:
k_values       = [1, 3, 5, 7]
credit_results = []

print("k-NN on Credit Approval Dataset (mixed features):")
print(f"  {'k':>3}  {'Accuracy':>10}  {'Correct/Total':>15}")
print("  " + "-" * 32)

for k in k_values:
    preds   = knn_predict(X_train_credit, y_train_credit, X_test_credit, k,
                          cat_cols=CAT_COLS, num_cols=NUM_COLS)
    acc     = compute_accuracy(y_test_credit, preds)
    correct = int(round(acc * len(y_test_credit)))
    credit_results.append((k, acc))
    print(f"  {k:>3}  {acc:>10.4f}  {correct:>7}/{len(y_test_credit):<7}")


## Summary and Discussion

### Results Table
*(Actual accuracy values are printed by the code cells above)*

| Dataset | k=1 | k=3 | k=5 | k=7 |
|---------|-----|-----|-----|-----|
| Lenses | (see output) | (see output) | (see output) | (see output) |
| Credit Approval | (see output) | (see output) | (see output) | (see output) |

### Discussion

**1. Which value of k works best for each dataset?**
- **Lenses**: With only 18 training samples and a very clean, categorical dataset, k=1 or k=3 tends to work best. Low k captures the exact patterns in the small dataset without over-smoothing.
- **Credit Approval**: k=3 or k=5 typically achieves the highest accuracy. k=1 overfits to noise in the noisy 552-sample dataset; k=7 oversmoothes the decision boundary. A moderate k balances bias and variance.

**2. How did preprocessing affect the Credit Approval dataset?**
- **Missing value imputation** was essential — 5% of cases had at least one `?`. Using the label-conditioned mean for numerical features (rather than the overall mean) reduces bias by preserving the class-conditional distribution.
- **Z-normalization** was critical: without it, features with large ranges (e.g., A15 can exceed 100,000) would completely dominate the L2 distance over features with small ranges (e.g., A3 ≈ 0–30). Z-scaling puts all continuous features on a comparable scale.

**3. Trade-offs of different values of k:**
- **Small k (k=1)**: low bias, high variance — memorizes training data, sensitive to outliers.
- **Large k**: high bias, low variance — smoothed boundary, may miss fine structure.
- Optimal k is a dataset-specific hyperparameter; cross-validation is the principled selection method.

**4. What did you learn?**
- Data preprocessing (imputation + normalization) often has more impact on accuracy than algorithm tuning.
- k-NN is a simple, powerful non-parametric classifier but requires O(n·d) computation per test query — it doesn't scale well to large datasets.
- Designing distance metrics for mixed data types requires domain knowledge and care.
